In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import re
import numpy as np


In [ ]:
def plot_training_folds(histories, dir=None, save_prefix='training_history'):
    plt.figure(figsize=(12, 4))

    # Validação - Loss
    plt.subplot(1, 2, 1)
    for i, hist in enumerate(histories):
        plt.plot(hist['val_loss'], label=f'Fold {i+1}')
    plt.title('Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Validação - Accuracy
    plt.subplot(1, 2, 2)
    for i, hist in enumerate(histories):
        val_acc_col = 'val_accuracy' if 'val_accuracy' in hist.columns else 'val_categorical_accuracy'
        plt.plot(hist[val_acc_col], label=f'Fold {i+1}')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    if dir:
        os.makedirs(dir, exist_ok=True)
        plt.savefig(os.path.join(dir, f'{save_prefix}_validation.png'))

    plt.show()
    plt.close('all')

    plt.figure(figsize=(12, 4))

    # Treinamento - Loss
    plt.subplot(1, 2, 1)
    for i, hist in enumerate(histories):
        plt.plot(hist['loss'], label=f'Fold {i+1}')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Treinamento - Accuracy
    plt.subplot(1, 2, 2)
    for i, hist in enumerate(histories):
        acc_col = 'accuracy' if 'accuracy' in hist.columns else 'categorical_accuracy'
        plt.plot(hist[acc_col], label=f'Fold {i+1}')
    plt.title('Training Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    if dir:
        plt.savefig(os.path.join(dir, f'{save_prefix}_training.png'))

    plt.show()
    plt.close('all')

# Função para calcular média e desvio padrão e formatar
def mean_std(arr):
    arr = np.array(arr)
    return f"{arr.mean():.4f} ± {arr.std(ddof=1):.4f}"

In [ ]:
dir_base = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/3D_BRAIN_NORM/results/folds"
folds_base_dir = f"{dir_base}/test_1"
folds_images = f"{folds_base_dir}/metrics"
os.makedirs(folds_images, exist_ok=True)

histories = []

for fold in os.listdir(folds_base_dir):
    fold_path = f"{folds_base_dir}/{fold}"
    if len(os.listdir(fold_path)) > 3:
        histories.append(pd.read_csv(f"{fold_path}/log_treino.csv"))

plot_training_folds(histories, folds_images, 'training_history')

In [ ]:
for fold in os.listdir(folds_base_dir):
    fold_path = f"{folds_base_dir}/{fold}"

    if len(os.listdir(fold_path)) > 3:
        with open(f'{fold_path}/val_classification_report.txt', 'r') as f:
            metrics = f.read()
        print(metrics)    

In [ ]:
# Classes e métricas que queremos extrair
classes = ['0', '1', '2', '3', '4', 'accuracy']
metrics_names = ['precision', 'recall', 'f1-score']

# Inicializar dicionário para armazenar métricas
data = {metric: {cls: [] for cls in classes} for metric in metrics_names}

# Para accuracy, só tem um valor por fold, não por classe, então guardaremos em recall para facilitar
accuracy_list = []

for fold in os.listdir(folds_base_dir):
    fold_path = os.path.join(folds_base_dir, fold)
    if os.path.isdir(fold_path) and len(os.listdir(fold_path)) > 3:
        file_path = os.path.join(fold_path, 'val_classification_report.txt')
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                report = f.read()

            # Encontrar as linhas que contêm métricas, exceto header e linhas vazias
            lines = [line.strip() for line in report.split('\n') if line.strip()]
            
            for line in lines:
                parts = re.split(r'\s{2,}', line)  # separar por 2 ou mais espaços
                if parts[0] in classes:
                    cls = parts[0]

                    if cls == 'accuracy':
                        # accuracy tem só um valor (na coluna precision para nosso caso)
                        accuracy_val = float(parts[1])
                        accuracy_list.append(accuracy_val)
                        # salvar accuracy no lugar especial
                        for metric in metrics_names:
                            data[metric]['accuracy'].append(accuracy_val)
                    else:
                        # para as outras classes e 'macro avg' e 'weighted avg'
                        # colunas são: precision, recall, f1-score, support (último)
                        try:
                            prec = float(parts[1])
                            rec = float(parts[2])
                            f1 = float(parts[3])
                            data['precision'][cls].append(prec)
                            data['recall'][cls].append(rec)
                            data['f1-score'][cls].append(f1)
                        except:
                            # caso formato diferente, ignorar
                            pass

# Salvar resultados no arquivo metrics_avg.txt
with open(f"{folds_images}/metrics_avg.txt", "w") as f_out:
    for metric in metrics_names:
        f_out.write(f"{metric.upper()}:\n")
        for cls in classes:
            # Pode ser que algumas listas estejam vazias, tratar isso
            values = data[metric][cls]
            if values:
                f_out.write(f"  Class {cls}: {mean_std(values)}\n")
        f_out.write("\n")

print("Média e desvio padrão calculados e salvos em metrics_avg.txt")
